# MSA29HCM - Natural Language Processing - Final Project

## Group Information

1. Hồ Nhật Thanh - 25MSA23226
2. Bùi Nguyễn Trúc Như - 25MSA23250
3. Nguyễn Văn Nhật - 25MSA23239

## I. Project Overview

### 1. Introduction and Objectives
The primary objective of this project is to implement, evaluate, and critically compare three distinct classes of Vector Space Models (VSM)—Sparse, Dense, and Contextual models—within a RAG framework for advanced information extraction tasks. By deconstructing the processing pipelines of each model, this project aims to demonstrate a profound mastery of modern retrieval architectures.

### 2. Model Architectures and Processing Pipelines
To thoroughly understand and master the retrieval process, this project orchestrates three distinct processing pipelines acting upon a unified, domain-specific corpus (an NLP technical document).

* **Phase 1: Data Ingestion and Chunking**

  The raw document is parsed, cleaned, and systematically divided into manageable text chunks using a sliding window approach (e.g., fixed token length with overlap) to ensure that localized context is preserved across boundaries.

* **Phase 2: Sparse Model Pipeline (Lexical Retrieval - BM25)**
  
  This pipeline represents traditional keyword-based search.

  * *Processing*: The corpus undergoes tokenization and normalization. The system builds an Inverted Index, mapping discrete vocabulary terms to their corresponding chunk IDs.

  * *Retrieval*: Upon receiving a query, the model calculates the Okapi BM25 score based on exact term frequency (TF) and inverse document frequency (IDF). It excels at exact-match lookups but is theoretically blind to semantic meaning.

* **Phase 3: Dense Model Pipeline (Semantic Retrieval - Bi-Encoder)**

  This pipeline addresses the vocabulary mismatch problem using deep learning.

  * *Processing*: Both the document chunks and the incoming query are passed independently through a Transformer-based Bi-Encoder (e.g., Sentence-BERT). The text is compressed into high-dimensional, dense floating-point vectors representing latent semantic meaning.

  * *Retrieval*: The system computes the Cosine Similarity between the query vector and all chunk vectors in the multidimensional space, enabling the retrieval of synonymous and conceptually related information.

* **Phase 4: Context Model Pipeline (Single-Stage Cross-Encoder)**
  
  This pipeline serves as the ultimate semantic precision baseline. Unlike industry-standard implementations that use a filtering step, this project intentionally deploys a "naive" or single-stage Cross-Encoder approach to empirically demonstrate both the absolute ceiling of retrieval accuracy and the severe computational limitations of deep learning inference.
  
  * *Processing*: No pre-computation or initial filtering is performed. Upon receiving a query, the system dynamically pairs the query with every single chunk present in the entire document corpus
  
  * *Retrieval*: The Cross-Encoder concatenates the user query and each individual chunk into a single, unified sequence (e.g., [CLS] Query [SEP] Chunk [SEP]). The Transformer's self-attention mechanism performs exhaustive token-level cross-comparisons across the entire dataset, calculating an absolute relevance score for every chunk to extract the definitive Top-K results
  
  * *Architectural Conclusion*: While this pure approach guarantees maximum precision by avoiding the information-loss bottleneck of Dense vectors, it is deliberately implemented to expose the catastrophic latency caused by the $O(L^2)$ time complexity of Transformers. By documenting this extreme execution time, the project mathematically proves the absolute necessity of adopting Two-Stage Retrieval architectures for real-time production systems.

#### **The Computational Imperative of Two-Stage Retrieval**

While the Cross-Encoder provides unparalleled semantic accuracy, applying it directly to an entire database (a Single-Stage approach) is mathematically and computationally unfeasible in production environments.

The core bottleneck lies in the Transformer architecture's self-attention mechanism, which possesses a quadratic time complexity of $O(L^2)$, where $L$ is the combined sequence length of the query and the chunk. Because the query and chunk must be processed simultaneously, Cross-Encoders cannot pre-compute and store document vectors in a database. Evaluating a single query against a modest corpus of 100,000 document chunks would require 100,000 distinct neural network inference passes at runtime, resulting in catastrophic latency (often taking minutes per query).

Therefore, implementing a Two-Stage Retrieval architecture is not merely an optimization, but a strict architectural requirement to balance speed and accuracy:
1. **Stage 1 (High Recall - Asymptotic Speed):**
The system utilizes the Dense or Sparse models (which operate on pre-computed Inverted Indexes or Approximate Nearest Neighbor graphs) to scan millions of records in milliseconds, filtering the corpus down to a small candidate pool (e.g., Top 50 chunks).
2. **Stage 2 (High Precision - Deep Inference):**
The Cross-Encoder acts strictly as a Re-ranker. It executes its computationally expensive cross-attention mechanism only on the 50 candidates retrieved in Stage 1.

This hybrid approach effectively resolves the $O(L^2)$ bottleneck, allowing the RAG system to deliver deep analytical accuracy while strictly adhering to enterprise latency constraints.

### 3. Testing Methodology
To empirically validate the theoretical behaviors of these models, a simple software testing framework is established. A curated "Ground Truth" dataset comprising varied test scenarios is executed across all three pipelines. The test cases are strategically categorized to expose the strengths and weaknesses of each architecture:

* **Exact Keyword Queries**: To test Sparse model precision.
* **Synonym and Semantic Queries**: To evaluate the Dense model's linguistic mapping.
* **Complex Contextual Traps**: To challenge Bi-Encoders and prove the necessity of Cross-Encoders.
* **Typo Robustness**: To test tokenizer resilience against malformed inputs.
* **Out-of-Domain Queries**: To verify the system's ability to reject irrelevant information.

### 4. Comparative Evaluation and Conclusions
The final phase of the project involves a quantitative and qualitative analysis of the test results.

* **Quantitative Metrics**: The performance of each model group is benchmarked using standard statistical retrieval metrics, primarily Mean Reciprocal Rank (MRR), to objectively measure ranking accuracy.

* **Architectural Assessment**: The project will analyze the trade-offs between computational latency, memory footprint, and retrieval precision.

# ĐỒ ÁN NLP: ĐÁNH GIÁ CÁC KIẾN TRÚC VECTOR SPACE MODEL TRONG RAG
**Mục tiêu:** Xây dựng và so sánh hiệu năng (MRR) của 3 kiến trúc truy xuất thông tin: Sparse Model (BM25), Dense Model (Bi-Encoder), và Context Model (Cross-Encoder) trên cùng một tập dữ liệu văn bản.

## Pipeline Architecture
1. **Data Ingestion:** Đọc văn bản thô, tiền xử lý và cắt nhỏ (Chunking).
2. **Indexing:** Khởi tạo các không gian vector và chỉ mục tìm kiếm.
3. **Retrieval Engine:** Định nghĩa 3 luồng tìm kiếm độc lập.
4. **Evaluation:** Chạy tự động tập Test Cases và tính điểm MRR.

In [ ]:
# Cài đặt các thư viện cần thiết (Bỏ comment khi chạy lần đầu trên Colab)
# !pip install sentence-transformers rank_bm25 pandas numpy tqdm

import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

# Import các thư viện NLP (Sẽ sử dụng sau)
# from rank_bm25 import BM25Okapi
# from sentence_transformers import SentenceTransformer, CrossEncoder

In [ ]:
def load_raw_data() -> str:
    """
    Tải dữ liệu văn bản thô.
    Trong demo này, có thể parse PDF hoặc dán trực tiếp text 3000 từ vào đây.
    """
    raise NotImplementedError("TODO: Khai báo biến chứa raw text hoặc viết hàm đọc file PDF.")
    return ""

def preprocess_and_chunk(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    Tiền xử lý text (xóa ký tự nhiễu) và cắt thành các đoạn nhỏ (chunks).
    Cần đảm bảo ngữ cảnh không bị đứt gãy giữa các chunks.

    Returns:
        List[str]: Danh sách các đoạn văn bản.
    """
    raise NotImplementedError("TODO: Triển khai logic cắt text (cố định độ dài hoặc theo câu).")
    return []

# --- Khởi tạo dữ liệu ---
# raw_text = load_raw_data()
# chunks = preprocess_and_chunk(raw_text)
# print(f"Total chunks created: {len(chunks)}")

In [ ]:
class SearchEngineModels:
    def __init__(self, chunks: List[str]):
        self.chunks = chunks
        self.bm25_index = None
        self.dense_model = None
        self.dense_embeddings = None
        self.cross_encoder = None

    def build_sparse_index(self):
        """Khởi tạo Inverted Index cho thuật toán BM25."""
        raise NotImplementedError("TODO: Khởi tạo BM25Okapi(tokenized_corpus).")

    def build_dense_index(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Tải Bi-Encoder và mã hóa toàn bộ chunks thành Dense Vectors."""
        raise NotImplementedError("TODO: Load SentenceTransformer và gọi model.encode(chunks).")

    def load_context_model(self, model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2'):
        """Tải Cross-Encoder (Không cần embed trước dữ liệu)."""
        raise NotImplementedError("TODO: Load CrossEncoder model.")

# --- Khởi tạo Models ---
# search_engine = SearchEngineModels(chunks)
# search_engine.build_sparse_index()
# search_engine.build_dense_index()
# search_engine.load_context_model()

In [ ]:
def retrieve_sparse(query: str, engine: SearchEngineModels, top_k: int = 5) -> List[Tuple[int, float]]:
    """
    Tìm kiếm bằng từ khóa (Lexical Search).
    Returns: List of tuples (chunk_id, score)
    """
    raise NotImplementedError("TODO: Tokenize query và dùng get_scores() của BM25.")
    return []

def retrieve_dense(query: str, engine: SearchEngineModels, top_k: int = 5) -> List[Tuple[int, float]]:
    """
    Tìm kiếm ngữ nghĩa bằng Bi-Encoder (Semantic Search).
    Returns: List of tuples (chunk_id, score)
    """
    raise NotImplementedError("TODO: Embed query và tính Cosine Similarity với dense_embeddings.")
    return []

def retrieve_context(query: str, engine: SearchEngineModels, top_k: int = 5) -> List[Tuple[int, float]]:
    """
    Tìm kiếm Two-Stage:
    1. Dùng retrieve_dense lấy Top 50.
    2. Dùng Cross-Encoder chấm điểm chéo (query, chunk) và Re-rank lấy Top 5.
    Returns: List of tuples (chunk_id, score)
    """
    raise NotImplementedError("TODO: Triển khai luồng Two-Stage Retrieval.")
    return []

In [ ]:
# Cấu trúc Test Case
test_cases = [
    {
        "query": "What is the formula for Okapi BM25 ranking algorithm?",
        "expected_chunk_id": 5 # ID của chunk chứa câu trả lời đúng (Cần xác định sau khi chunking)
    },
    # ... Thêm 19 test cases còn lại vào đây ...
]

def calculate_mrr(retrieved_ids: List[int], expected_id: int) -> float:
    """
    Tính điểm Mean Reciprocal Rank cho 1 câu truy vấn.
    Nếu expected_id nằm ở vị trí i (1-based index), điểm = 1/i.
    Nếu không có, điểm = 0.
    """
    raise NotImplementedError("TODO: Viết logic tìm rank của expected_id trong retrieved_ids.")
    return 0.0

def run_evaluation(test_cases: List[Dict], engine: SearchEngineModels):
    """
    Chạy toàn bộ test cases qua 3 mô hình và in ra điểm MRR trung bình.
    """
    results = {
        "Sparse (BM25)": [],
        "Dense (Bi-Encoder)": [],
        "Context (Cross-Encoder)": []
    }

    raise NotImplementedError("TODO: Lặp qua test_cases, gọi 3 hàm retrieve_*, tính MRR và lưu vào results.")

    # Gợi ý đầu ra cuối cùng:
    # mrr_sparse = sum(results["Sparse (BM25)"]) / len(test_cases)
    # print(f"MRR Sparse: {mrr_sparse:.4f}")